In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import Dataset
import os
import numpy as np
import cv2
from torch.utils.data import DataLoader
from C3D import C3D
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime
import socket
import timeit
from tqdm import tqdm
import torch.optim as optim


class VideoDataset(Dataset):
    def __init__(self, dataset_path, images_path, clip_len):
        self.dataset_path = dataset_path
        self.images_path = images_path
        self.clip_len = clip_len

        # 后续数据预处理的值
        self.resize_height = 128
        self.resize_width = 171
        self.crop_size = 112

        # 读取对应训练集/验证集/测试集下的各种类别的行为动作
        # 每个行为动作下的视频已经被处理成单个图片数据
        # 将对应动作的数据的文件名作为标签保存到labels列表中，对应的动作数据集的路径保存到self.fnames列表中，标签和数据是一一对应状态
        folder = os.path.join(self.dataset_path, images_path)
        # print(folder)  # data/train
        self.fnames, labels = [], []
        for label in sorted(os.listdir(folder)):
            for fname in os.listdir(os.path.join(folder, label)):
                self.fnames.append(os.path.join(folder, label, fname))
                labels.append(label)
        # print(self.fnames) # ['./data\\train\\ApplyEyeMakeup\\v_ApplyEyeMakeup_g01_c01',...]
        # print(labels)  # ['ApplyEyeMakeup', 'ApplyEyeMakeup', 'ApplyEyeMakeup', ...]
        print("Number of {} videos: {:d}".format(images_path, len(self.fnames)))
        self.label2index = {
            label: index for index, label in enumerate(sorted(set(labels)))
        }
        print(
            self.label2index
        )  # {'ApplyEyeMakeup': 0, 'Archery': 1, 'BalanceBeam': 2, ...}
        self.label_array = np.array(
            [self.label2index[label] for label in labels], dtype=int
        )
        print(self.label_array)  # [0 0 0 ... 4 4 4]

    def __len__(self):
        return len(self.fnames)

    def __getitem__(self, index):
        buffer = self.load_frames(
            self.fnames[index]
        )  # 加载对应类别的动作数据集，转换为（帧数，高度，宽度，通道数）的4维张量
        buffer = self.crop(
            buffer, self.clip_len, self.crop_size
        )  # 随机裁剪视频帧，转换为（clip_len，高度，宽度，通道数）的4维张量
        buffer = self.normalize(buffer)  # 归一化视频帧
        buffer = self.to_tensor(buffer)  # 转换为PyTorch张量
        label = np.array(self.label_array[index], dtype=np.int8)  # 获取对应索引的标签

        # 数据形状为（通道数，clip_len，高度，宽度）的4维张量
        # 标签形状为（1，）的1维张量
        return torch.from_numpy(buffer), torch.from_numpy(label)  # 返回视频帧张量和标签

    def load_frames(self, file_dir):
        # 将文件夹下的图片文件进行排序
        frames = sorted([os.path.join(file_dir, img) for img in os.listdir(file_dir)])
        # 获取该文件夹下的图片文件数量
        frame_count = len(frames)
        # 生成一个空的（frame_count，高度，宽度，通道数）的4维张量
        buffer = np.empty(
            (frame_count, self.resize_height, self.resize_width, 3), dtype=np.float32
        )
        # 遍历文件夹下的所有图片文件
        for i, frame_name in enumerate(frames):
            # 读取图片文件
            frame = np.array(cv2.imread(frame_name)).astype(np.float64)
            # 将图片数据存储到buffer张量中（按照帧数的维度存储图片）
            buffer[i] = frame
        return buffer

    # 裁剪出（16，112，112，3）的视频帧张量
    def crop(self, buffer, clip_len, crop_size):
        time_index = np.random.randint(
            buffer.shape[0] - clip_len
        )  # 生成一个深度方向上的随机长度
        height_index = np.random.randint(
            buffer.shape[1] - crop_size
        )  # 生成一个高度方向上的随机长度
        width_index = np.random.randint(
            buffer.shape[2] - crop_size
        )  # 生成一个宽度方向上的随机长度

        # 从buffer张量中提取随机裁剪的视频帧（按照clip_len的长度）
        cropped_buffer = buffer[
            time_index : time_index + clip_len,
            height_index : height_index + crop_size,
            width_index : width_index + crop_size,
            :,
        ]
        return cropped_buffer

    def normalize(self, buffer):
        # 归一化视频帧（将像素值缩放到[-1, 1]范围）
        for i, frame in enumerate(buffer):
            frame -= np.array([[[90.0, 98.0, 102.0]]])  # 减去每个通道的均值
            buffer[i] = frame

        return buffer

    def to_tensor(self, buffer):
        return buffer.transpose(
            (3, 0, 1, 2)
        )  # 转换为（通道数，clip_len，高度，宽度）的4维张量


def train_model(
    num_epochs,
    num_classes,
    lr,
    device,
    save_dir,
    train_dataloader,
    valid_dataloader,
    test_dataloader,
):
    model = C3D(num_classes=num_classes)

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-5)

    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

    model = model.to(device)
    criterion = criterion.to(device)

    # 日志记录
    log_dir = os.path.join(
        save_dir,
        "logs",
        datetime.now().strftime("%b%d_%H-%M-%S") + "_" + socket.gethostname(),
    )
    writer = SummaryWriter(log_dir)

    trainval_loaders = {"train": train_dataloader, "val": valid_dataloader}
    trainval_sizes = {
        x: len(trainval_loaders[x].dataset) for x in ["train", "val"]
    }  # 计算训练集和验证集的大小
    test_size = len(test_dataloader.dataset)  # 计算测试集的大小

    for epoch in range(num_epochs):
        for phase in ["train", "val"]:
            start_time = timeit.default_timer()
            running_loss = 0.0  # 统计这一轮的所有批次的平均损失值之和
            running_corrects = 0  # 这一轮预测正确的个数

            if phase == "train":
                model.train()
            else:
                model.eval()

            for inputs, labels in tqdm(trainval_loaders[phase]):
                # 将数据和标签放入到设备中
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                if phase == "train":
                    outputs = model(inputs)
                else:
                    with torch.no_grad():
                        outputs = model(inputs)

                probs = nn.Softmax(dim=1)(outputs)
                # print(probs.shape) # (batch_size, 101)
                result = torch.max(probs, dim=1)
                # result 是一个元组：(values, indices)
                # values 是每个样本在101个类别的概率最大值(batch_size,)
                # indices 是每个样本在101个类别的概率最大值的索引（即预测类别） (batch_size,)
                preds = result.indices  # 样本的预测结果，降维成1维了
                # print(preds.shape)  # (batch_size,)
                labels = labels.long()  # 真实标签,转换为int64类型
                # print(labels.shape)  # (batch_size,)
                loss = criterion(outputs, labels)

                # 反向传播和优化
                if phase == "train":
                    loss.backward()
                    optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            if phase == "train":
                scheduler.step()

            epoch_loss = (
                running_loss / trainval_sizes[phase]
            )  # 计算该轮次的损失值，总loss除以样本数量
            epoch_acc = (
                running_corrects.double() / trainval_sizes[phase]
            )  # 计算该轮次的准确率，正确预测的个数除以样本数量

            writer.add_scalar(phase + "_loss", epoch_loss, epoch)
            writer.add_scalar(phase + "_acc", epoch_acc, epoch)

            print(
                "Epoch:{}/{} Loss: {:.4f} Acc: {:.4f} Time: {:.2f}s".format(
                    epoch,
                    phase,
                    epoch_loss,
                    epoch_acc,
                    timeit.default_timer() - start_time,
                )
            )
    writer.close()

    torch.save(
        {
            "epoch": epoch + 1,
            "state_dict": model.state_dict(),
            "opt_dict": optimizer.state_dict(),
        },
        os.path.join(
            save_dir, "models", "C3D" + "_epoch-" + str(epoch + 1) + ".pth.tar"
        ),
    )


if __name__ == "__main__":
    train_dataset = VideoDataset(
        dataset_path="./data",
        images_path="train",
        clip_len=16,
    )
    valid_dataset = VideoDataset(
        dataset_path="./data",
        images_path="valid",
        clip_len=16,
    )
    test_dataset = VideoDataset(
        dataset_path="./data",
        images_path="test",
        clip_len=16,
    )

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    valid_loader = DataLoader(valid_dataset, batch_size=64, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

    # print(train_loader)
    # for inputs, labels in tqdm(train_loader):
    #     print(inputs.shape)  # (64, 3, 16, 112, 112)
    #     print(labels.shape)  # (64,)
    #     break
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    num_epochs = 10
    num_classes = 101
    lr = 1e-3
    save_dir = "./data"
    train_model(
        num_epochs,
        num_classes,
        lr,
        device,
        save_dir,
        train_loader,
        valid_loader,
        test_loader,
    )

Number of train videos: 8460
{'ApplyEyeMakeup': 0, 'ApplyLipstick': 1, 'Archery': 2, 'BabyCrawling': 3, 'BalanceBeam': 4, 'BandMarching': 5, 'BaseballPitch': 6, 'Basketball': 7, 'BasketballDunk': 8, 'BenchPress': 9, 'Biking': 10, 'Billiards': 11, 'BlowDryHair': 12, 'BlowingCandles': 13, 'BodyWeightSquats': 14, 'Bowling': 15, 'BoxingPunchingBag': 16, 'BoxingSpeedBag': 17, 'BreastStroke': 18, 'BrushingTeeth': 19, 'CleanAndJerk': 20, 'CliffDiving': 21, 'CricketBowling': 22, 'CricketShot': 23, 'CuttingInKitchen': 24, 'Diving': 25, 'Drumming': 26, 'Fencing': 27, 'FieldHockeyPenalty': 28, 'FloorGymnastics': 29, 'FrisbeeCatch': 30, 'FrontCrawl': 31, 'GolfSwing': 32, 'Haircut': 33, 'HammerThrow': 34, 'Hammering': 35, 'HandstandPushups': 36, 'HandstandWalking': 37, 'HeadMassage': 38, 'HighJump': 39, 'HorseRace': 40, 'HorseRiding': 41, 'HulaHoop': 42, 'IceDancing': 43, 'JavelinThrow': 44, 'JugglingBalls': 45, 'JumpRope': 46, 'JumpingJack': 47, 'Kayaking': 48, 'Knitting': 49, 'LongJump': 50, 'Lun

100%|██████████| 133/133 [2:19:57<00:00, 63.14s/it] 


Epoch:0/train Loss: 4.6139 Acc: 0.0125 Time: 8397.66s


100%|██████████| 34/34 [14:17<00:00, 25.22s/it]


Epoch:0/val Loss: 4.6105 Acc: 0.0185 Time: 857.43s
